# Fine-tuning do detector de infrações: pré-anotação → revisão → treino

Este notebook é um roteiro de estudo **e** de trabalho. A ideia é você rodar célula por célula, ler o porquê de cada passo e fazer os exercícios marcados com ✍️ antes de seguir.

## O plano

```
dataset atual (1 caixa por foto: só a infração que foi enviada ao backend)
   │  1. o modelo antigo roda em todas as fotos
   ▼
pré-anotações (caixa confirmada + sugestões do modelo antigo)
   │  2. VOCÊ revisa: corrige classe, apaga falso positivo, desenha o que faltou
   ▼
dataset revisado (todas as instâncias de todas as classes rotuladas)
   │  3. separação treino / validação
   ▼
fine-tuning a partir dos pesos do modelo antigo  →  modelo novo
   │  4. avaliação: antigo x novo no mesmo conjunto de validação
```

## Por que rotular *tudo* na imagem importa

Um detector YOLO aprende duas coisas ao mesmo tempo: "aqui tem um objeto da classe X" e "aqui **não** tem nada" (fundo). Tudo que **não** tem caixa é tratado como fundo.

Pense numa foto com três motociclistas sem capacete e só um rotulado. Se o modelo acertar os outros dois, o treino **pune** ele por isso, porque para o dataset aquilo é fundo. Com milhares de fotos assim, o modelo recebe sinais contraditórios e passa a detectar menos (a **recall** cai). É por isso que esse trabalho de revisão vale a pena.

## Vocabulário rápido

- **Pré-anotação** (*model-assisted labeling*): usar um modelo para gerar rótulos iniciais que um humano corrige. Corrigir é bem mais rápido que desenhar do zero.
- **Transfer learning / fine-tuning**: em vez de começar com pesos aleatórios, o treino parte de pesos que já sabem algo útil (o seu modelo antigo) e ajusta esses pesos com dados novos. Converge mais rápido e precisa de menos dados.
- **Época** (*epoch*): uma passada completa por todas as imagens de treino.
- **mAP**: a métrica principal de detecção. Vamos destrinchar ela na seção 8.

> **Legenda das células:** 🔁 = rode sempre que reabrir o notebook (define variáveis e funções) · ⚠️ = rode uma vez só (gera arquivos, é lento ou sobrescreve coisas).

## 0. Ambiente

Rode estes comandos **no terminal** (não no notebook), na pasta do projeto. O PyTorch precisa ser instalado **antes** do `ultralytics`, pelo índice CUDA. Se você instalar só o `ultralytics`, o pip baixa o PyTorch de CPU e o treino fica umas 20x mais lento.

```powershell
python -m venv .venv
.venv\Scripts\activate
pip install torch torchvision --index-url https://download.pytorch.org/whl/cu126
pip install ultralytics pandas ipykernel onnx onnxruntime fastapi uvicorn
```

Use `onnxruntime` (CPU), e não `onnxruntime-gpu`: a versão GPU atual exige CUDA 13, e o PyTorch acima usa CUDA 12.6. O ONNX só é usado para ler os pesos e conferir a reconstrução (seção 3), então CPU basta.

Depois, no canto superior direito do notebook, escolha o kernel **.venv**.

O modelo de produção (`fiscalizer.onnx`) fica na raiz do projeto. Na seção 3, vamos transformá-lo num `.pt` treinável.

In [ ]:
# 🔁 Confere se a GPU está visível para o PyTorch
import torch, ultralytics

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__)
print("CUDA disponível:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"| {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Configuração e funções auxiliares

Tudo que o notebook gera fica em pastas separadas: `work/`, `datasets/` e `runs/`. **O dataset original nunca é alterado.** Assim, se algo der errado, é só apagar a pasta gerada e rodar de novo.

As imagens não são copiadas: usamos *hard links*, que são outra "entrada" para o mesmo arquivo no disco. Assim os 1,4 GB de fotos não ocupam espaço em dobro.

In [ ]:
# 🔁 Caminhos e constantes
from pathlib import Path

ROOT = Path.cwd()
DATASET_ORIGINAL = ROOT / "infractions-yolo-2026-09-24T21-32-33-882Z"
MODELO_ONNX = ROOT / "fiscalizer.onnx"                           # modelo de produção (só inferência)
MODELO_ANTIGO = ROOT / "models" / "fiscalizer_reconstruido.pt"   # gerado a partir do ONNX na seção 3

WORK = ROOT / "work"
REVISAO = WORK / "revisao"          # pré-anotações que você vai revisar
REVISADO = WORK / "revisado"        # rótulos que voltam da ferramenta de anotação (CVAT)
DATASET_FINAL = ROOT / "datasets" / "infractions-v2"
RUNS = ROOT / "runs"

SEED = 42  # fixa a aleatoriedade: mesmo split e mesmas amostras toda vez

assert DATASET_ORIGINAL.exists(), DATASET_ORIGINAL
print("ONNX encontrado:", MODELO_ONNX.exists(), "| .pt reconstruído:", MODELO_ANTIGO.exists())

In [ ]:
# 🔁 Funções auxiliares: ler/escrever rótulos YOLO, desenhar caixas, IoU
import hashlib, json, os, random, shutil, zipfile
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
from PIL import Image, ImageDraw


def ler_labels(caminho):
    """Lê um .txt YOLO -> lista de (classe, x_centro, y_centro, largura, altura)."""
    caminho = Path(caminho)
    if not caminho.exists():
        return []
    caixas = []
    for linha in caminho.read_text().splitlines():
        partes = linha.split()
        if partes:
            caixas.append((int(partes[0]), *map(float, partes[1:5])))
    return caixas


def escrever_labels(caminho, caixas):
    """Escreve caixas (classe, x, y, w, h[, conf]) no formato YOLO. A confiança é descartada."""
    linhas = [f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}" for c, x, y, w, h, *_ in caixas]
    Path(caminho).write_text("\n".join(linhas) + ("\n" if linhas else ""))


def iou(a, b):
    """Intersection over Union entre duas caixas (x_centro, y_centro, w, h) normalizadas."""
    ax1, ay1, ax2, ay2 = a[0] - a[2] / 2, a[1] - a[3] / 2, a[0] + a[2] / 2, a[1] + a[3] / 2
    bx1, by1, bx2, by2 = b[0] - b[2] / 2, b[1] - b[3] / 2, b[0] + b[2] / 2, b[1] + b[3] / 2
    inter = max(0, min(ax2, bx2) - max(ax1, bx1)) * max(0, min(ay2, by2) - max(ay1, by1))
    uniao = a[2] * a[3] + b[2] * b[3] - inter
    return inter / uniao if uniao > 0 else 0.0


def linkar(src, dst):
    """Hard link (não ocupa espaço extra); se o sistema não permitir, copia."""
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)


CORES = ["#e6194b", "#3cb44b", "#ffe119", "#4363d8", "#f58231", "#911eb4",
         "#46f0f0", "#f032e6", "#bcf60c", "#fabebe", "#008080", "#9a6324",
         "#aaffc3", "#800000", "#808000", "#000075"]


def desenhar(img_path, caixas, nomes, ax, titulo=""):
    """Desenha caixas YOLO numa imagem. Se a caixa tiver um 6º valor, ele é mostrado como confiança."""
    img = Image.open(img_path).convert("RGB")
    W, H = img.size
    d = ImageDraw.Draw(img)
    for c, x, y, w, h, *resto in caixas:
        cor = CORES[c % len(CORES)]
        x1, y1, x2, y2 = (x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H
        d.rectangle([x1, y1, x2, y2], outline=cor, width=4)
        texto = str(nomes[c]) + (f" {resto[0]:.2f}" if resto else "")
        d.text((x1 + 4, max(0, y1 - 30)), texto, fill=cor, font_size=28)
    ax.imshow(img)
    ax.set_title(titulo, fontsize=9)
    ax.axis("off")


def grade(n, cols=3, tamanho=(6, 3.6)):
    linhas = (n + cols - 1) // cols
    fig, axs = plt.subplots(linhas, cols, figsize=(tamanho[0] * cols, tamanho[1] * linhas))
    axs = np.atleast_1d(axs).ravel()
    for ax in axs[n:]:
        ax.axis("off")
    return fig, axs

## 2. Conhecendo o dataset

Antes de treinar qualquer coisa, **olhe os dados**. Boa parte dos problemas de modelo são, na verdade, problemas de dados.

Formato YOLO, um `.txt` por imagem, uma linha por objeto:

```
classe x_centro y_centro largura altura      ← tudo normalizado entre 0 e 1
2      0.674219 0.472222 0.056771 0.118519
```

In [ ]:
# 🔁 Carrega os rótulos originais numa tabela
with open(DATASET_ORIGINAL / "data.yaml", encoding="utf-8") as f:
    NOMES_ORIGINAIS = yaml.safe_load(f)["names"]   # {id: nome}

registros = []
for txt in sorted((DATASET_ORIGINAL / "labels").glob("*.txt")):
    for c, x, y, w, h in ler_labels(txt):
        registros.append(dict(stem=txt.stem, classe=NOMES_ORIGINAIS[c], x=x, y=y, w=w, h=h))
df_orig = pd.DataFrame(registros)

print(f"{df_orig.stem.nunique()} imagens, {len(df_orig)} caixas")
print("Classes no data.yaml:", NOMES_ORIGINAIS)
df_orig.classe.value_counts().to_frame("caixas")

**Repare:** o `data.yaml` lista 7 classes, mas só 6 aparecem nos rótulos. A classe `Monitoramento de Faixa de Ônibus (IA)` tem cerca de 3.300 linhas no `metadata.csv`, mas nenhuma imagem veio na exportação. A célula abaixo confirma isso. Se você quiser essa classe no modelo novo, vai precisar reexportar essas fotos do backend.

Repare também no desbalanceamento: `cellphone_infraction` tem só ~60 exemplos contra ~1.400 de outras classes. Espere um desempenho pior nela.

In [ ]:
meta = pd.read_csv(DATASET_ORIGINAL / "metadata.csv", sep=";", encoding="utf-8")
meta["imagem_existe"] = meta["image"].map(lambda n: (DATASET_ORIGINAL / "images" / n).exists())
meta.groupby("class")["imagem_existe"].agg(linhas="size", com_imagem="sum")

### Tamanho dos objetos

O YOLO redimensiona a imagem para `imgsz` (ex.: 640 px no lado maior) antes de processar. Um objeto de 100 px numa foto 1920x1080 vira ~33 px em 640. Objetos pequenos demais depois do redimensionamento são difíceis de detectar. Isso vai orientar a escolha do `imgsz` no treino.

In [ ]:
W_IMG, H_IMG = Image.open(next((DATASET_ORIGINAL / "images").glob("*.jpg"))).size
tam = df_orig.assign(
    largura_px=df_orig.w * W_IMG,
    altura_px=df_orig.h * H_IMG,
    largura_em_640=df_orig.w * 640,
    largura_em_960=df_orig.w * 960,
)
print(f"Resolução das imagens: {W_IMG}x{H_IMG}")
tam.groupby("classe")[["largura_px", "altura_px", "largura_em_640", "largura_em_960"]].median().round(0)

In [ ]:
# Amostra visual: rode várias vezes (ou mude a classe) para ter uma noção do dataset
CLASSE_PARA_VER = "no_hand"
N = 6

stems = df_orig[df_orig.classe == CLASSE_PARA_VER].stem.sample(N).tolist()
fig, axs = grade(N)
for stem, ax in zip(stems, axs):
    desenhar(DATASET_ORIGINAL / "images" / f"{stem}.jpg",
             ler_labels(DATASET_ORIGINAL / "labels" / f"{stem}.txt"), NOMES_ORIGINAIS, ax, stem)
plt.tight_layout()

> ✍️ **Exercício 1.** Olhe umas 20 fotos de classes diferentes e anote:
> 1. Em quantas existe **outra** infração visível que não está rotulada? Isso dá uma ideia do tamanho do problema.
> 2. As caixas de uma mesma classe seguem um padrão? Por exemplo, `no_helmet` marca a cabeça ou a moto inteira?
> 3. `WRONG_WAY` (contramão) vai ser descartada: ela não é algo que se vê numa foto, e sim uma regra do backend (um `car_front` ou `car_back` numa via específica). Olhe algumas fotos dela mesmo assim: o veículo está lá, e agora ele precisa receber a caixa de `car_front`/`car_back`.
>
> Guarde as respostas do item 2: elas viram o seu **manual de anotação** na seção 5.

## 3. O modelo antigo

### `.pt`, `.onnx` e `.engine`: qual a diferença?

| formato | o que é | treina? |
|---|---|---|
| `.pt` | checkpoint do PyTorch: arquitetura, pesos, BatchNorm separado e os argumentos de treino | **sim**, é o único |
| `.onnx` | grafo num formato aberto, feito para inferência. Roda em qualquer lugar (onnxruntime, OpenCV, TensorRT). O BatchNorm é *fundido* nas convoluções e a entrada tem tamanho fixo | não |
| `.engine` | TensorRT compilado para **uma** GPU e **uma** versão do TensorRT. É o mais rápido, mas não é portável e provavelmente nem roda nesta máquina. Vamos ignorá-lo | não |

Ou seja: para treinar precisamos de um `.pt`. O caminho ideal é conseguir o `best.pt` original. O próprio ONNX diz onde ele foi treinado (rode a célula abaixo); normalmente fica em `runs/detect/<nome>/weights/best.pt` nessa máquina. Se conseguir, coloque em `models/`, aponte `MODELO_ANTIGO` para ele e pule a seção 3.2.

Enquanto isso, dá para **reconstruir** um `.pt` a partir do ONNX.

In [ ]:
# 🔁 O que o ONNX conta sobre si mesmo (o ultralytics grava isso no export)
from onnx_para_pt import metadados

meta_onnx = metadados(MODELO_ONNX)
for k in ("description", "date", "version", "imgsz", "batch", "args"):
    print(f"{k:>12}: {meta_onnx[k]}")
print("\nClasses:")
for i, n in meta_onnx["names"].items():
    print(f"  {i:>2} {n}")

### 3.2 Reconstruindo um `.pt` treinável a partir do ONNX

O export do ultralytics mantém os nomes dos pesos do PyTorch (`model.0.conv.weight`, `model.0.conv.bias`, ...). Pelo formato do primeiro peso (16 filtros) e pelo total de parâmetros (3,05 milhões), a arquitetura é uma **YOLOv8n**. A ideia é criar uma YOLOv8n vazia e copiar os pesos camada por camada. A lógica está em `onnx_para_pt.py`; vale a leitura.

**O detalhe difícil é o BatchNorm.** No treino, cada bloco faz `conv → BatchNorm → SiLU`:

\[ y = \gamma \cdot \frac{W * x - \mu}{\sqrt{\sigma^2 + \epsilon}} + \beta \]

No export, isso vira uma convolução só, com peso \(W' = W \cdot \gamma / \sqrt{\sigma^2+\epsilon}\) e bias \(b' = \beta - \gamma\mu / \sqrt{\sigma^2+\epsilon}\). Os valores originais de \(\gamma, \beta, \mu, \sigma^2\) se perdem: infinitas combinações dão o mesmo \(W', b'\).

A saída é escolher uma combinação que funcione. Copiamos \(W'\) para a conv e ajustamos o BN para que \(\text{BN}(z) = z + b'\):
- \(\mu, \sigma^2\) = média e variância **reais** da saída da conv, medidas em 64 fotos do dataset;
- \(\gamma = \sqrt{\sigma^2+\epsilon}\) e \(\beta = \mu + b'\).

Em modo de avaliação, o resultado é **exatamente** o modelo original. No treino, o BN usa a média e a variância de cada lote em vez de \(\mu, \sigma^2\). Como calibramos com fotos reais, esses valores ficam próximos, e o modelo parte praticamente de onde estava.

> **Versões importam.** Na primeira tentativa a reconstrução divergia a partir da camada 9 (SPPF). O modelo foi treinado no ultralytics 8.4.47, e versões mais novas passaram a adicionar uma ativação SiLU na primeira conv do SPPF de YAMLs antigos. O `onnx_para_pt.py` lê o grafo do ONNX, descobre quais convs têm ativação e monta a arquitetura igual. Moral: ao reaproveitar modelos, sempre **prove** a equivalência numericamente, como na célula depois da próxima.

In [ ]:
# ⚠️ Reconstrói o .pt (menos de 1 min)
from onnx_para_pt import reconstruir, salvar

imagens = sorted((DATASET_ORIGINAL / "images").glob("*.jpg"))
modelo_rec, meta_onnx = reconstruir(MODELO_ONNX, random.Random(SEED).sample(imagens, 64))
MODELO_ANTIGO.parent.mkdir(exist_ok=True)
salvar(modelo_rec, meta_onnx, MODELO_ANTIGO)
print("Salvo em", MODELO_ANTIGO)

In [ ]:
# Prova de equivalência: a MESMA entrada 640x640 nos dois modelos
from onnx_para_pt import preprocessar
from ultralytics import YOLO

m_pt, m_onnx = YOLO(MODELO_ANTIGO), YOLO(str(MODELO_ONNX), task="detect")
amostra = random.Random(1).sample(sorted((DATASET_ORIGINAL / "images").glob("*.jpg")), 30)
iguais = caixas = 0
for p in amostra:
    x = preprocessar(p)
    a = m_pt.predict(x, conf=0.25, device="cpu", verbose=False)[0].boxes
    b = m_onnx.predict(x, conf=0.25, device="cpu", verbose=False)[0].boxes
    caixas += len(b)
    iguais += a.cls.tolist() == b.cls.tolist() and np.allclose(a.xyxy.numpy(), b.xyxy.numpy(), atol=0.5)
print(f"{iguais}/{len(amostra)} imagens com detecções idênticas ({caixas} caixas)")

Por que passar um tensor pronto (`preprocessar`) em vez do caminho da imagem? Com um `.pt`, o ultralytics redimensiona para o menor retângulo múltiplo de 32 (ex.: 640x384). O ONNX tem entrada fixa 640x640. Com entradas diferentes, as detecções mudam um pouco, e aqui queremos comparar **os modelos**, não o pré-processamento.

Daqui para frente, o "modelo antigo" é o `.pt` reconstruído: roda na GPU, aceita qualquer tamanho de lote e pode ser treinado.

In [ ]:
# 🔁 Carrega o modelo antigo (.pt) e mostra o que ele sabe detectar
from ultralytics import YOLO

modelo_antigo = YOLO(MODELO_ANTIGO)
args_treino = (modelo_antigo.ckpt or {}).get("train_args", {}) if hasattr(modelo_antigo, "ckpt") else {}
IMGSZ_ANTIGO = args_treino.get("imgsz", 640)

print("Classes do modelo antigo:", modelo_antigo.names)
print("imgsz usado no treino antigo:", IMGSZ_ANTIGO)
print("Arquitetura:", args_treino.get("model", "?"), "| épocas:", args_treino.get("epochs", "?"))

### Decidindo as classes finais

O modelo antigo tem 13 classes, em três grupos:

| grupo | classes | no dataset atual |
|---|---|---|
| infrações | `cellphone_infraction`, `no_hand`, `no_helmet`, `no_rearview`, `no_seatbelt`, `sandals_infraction`, `infraction_ontruck` | as 5 primeiras têm caixas confirmadas; `sandals` e `ontruck` só vêm das sugestões |
| "positivas" | `helmet`, `seatbelt`, `has_rearview`, `shoes` | nenhuma caixa, só sugestões |
| veículos | `car_front`, `car_back` | nenhuma caixa, só sugestões |

**Regra de ouro:** toda classe da lista precisa ser rotulada em **todas** as imagens. Se `helmet` estiver na lista mas você só revisar as caixas de infração, os capacetes sem caixa viram "fundo", e o problema do começo volta.

**Opção A (padrão abaixo):** exatamente as 13 classes do modelo antigo, **na mesma ordem**.
- Os ids e o formato da saída não mudam, então o modelo novo entra no lugar do antigo sem mexer no backend.
- O fine-tuning reaproveita 100% da rede, inclusive a camada de classificação.
- É a opção que dá mais trabalho de revisão: todo capacete, sapato e carro de cada foto precisa ser conferido. A pré-anotação faz a maior parte.

**Opção B:** só as infrações. Dá menos trabalho, mas perde duas coisas: as classes "positivas" ensinam contraste (ver `helmet` e `no_helmet` ajuda o modelo a entender a diferença), e se o backend usa `car_front`/`car_back` para algo, você quebra essa parte.

Se tirar classes, tire a classe inteira, e não "algumas caixas" dela.

**Sobre a contramão (`WRONG_WAY`):** o dataset tem essa classe, mas ela é uma **regra do backend**, e não um objeto: um `car_front` ou `car_back` detectado numa via específica. Por isso a caixa confirmada dela é descartada (`None` no `MAPA_DATASET`). As 492 fotos **continuam** no dataset: o veículo delas recebe a sugestão de `car_front`/`car_back` do modelo antigo, que você confere na revisão. Não dá para mapear direto para `car_front`, porque a caixa confirmada não diz se o carro estava de frente ou de costas.

In [ ]:
# 🔁 ✍️ EDITE AQUI: lista final de classes e como as classes antigas se traduzem nela

CLASSES_FINAIS = list(modelo_antigo.names.values())

# nome no modelo antigo -> nome em CLASSES_FINAIS (ou None para descartar as sugestões dessa classe).
# Classes com o mesmo nome nos dois lados não precisam aparecer aqui.
MAPA_ANTIGO = {
    # "shoes": None,
}

# nome no dataset original -> nome em CLASSES_FINAIS (ou None para descartar a caixa confirmada)
MAPA_DATASET = {
    "Monitoramento de Faixa de Ônibus (IA)": None,   # sem imagens na exportação
    "WRONG_WAY": None,                               # regra do backend sobre car_front/car_back, não um objeto
}


def resolver(nome, mapa):
    if nome in mapa:
        return mapa[nome]
    return nome if nome in CLASSES_FINAIS else None


MAPA_ANTIGO_ID = {}
for id_antigo, nome in modelo_antigo.names.items():
    destino = resolver(nome, MAPA_ANTIGO)
    MAPA_ANTIGO_ID[id_antigo] = CLASSES_FINAIS.index(destino) if destino else None
    print(f"modelo antigo  {id_antigo:>2} {nome:<28} -> {destino or '(descartada)'}")

MAPA_ORIGINAL_ID = {}
print()
for i, nome in NOMES_ORIGINAIS.items():
    destino = resolver(nome, MAPA_DATASET)
    MAPA_ORIGINAL_ID[i] = CLASSES_FINAIS.index(destino) if destino else None
    print(f"dataset        {i:>2} {nome:<28} -> {destino or '(descartada)'}")

print("\nCLASSES_FINAIS:", dict(enumerate(CLASSES_FINAIS)))
sem_fonte = set(CLASSES_FINAIS) - {resolver(n, MAPA_ANTIGO) for n in modelo_antigo.names.values()}
if sem_fonte:
    print("⚠️ Classes finais que o modelo antigo NÃO sugere (vão depender só de você na revisão):", sem_fonte)


def gt_final(stem):
    """Caixa confirmada do dataset original, já com o id da lista final."""
    return [(MAPA_ORIGINAL_ID[c], x, y, w, h)
            for c, x, y, w, h in ler_labels(DATASET_ORIGINAL / "labels" / f"{stem}.txt")
            if MAPA_ORIGINAL_ID.get(c) is not None]

## 4. Pré-anotação

Para cada imagem, juntamos:
1. a **caixa confirmada** (veio do backend e foi validada por humano, então é "verdade"), e
2. as **sugestões** do modelo antigo.

Se uma sugestão **da mesma classe** cai em cima da caixa confirmada (IoU > 0,5), ela é descartada: é a mesma detecção. Se for de **outra classe** no mesmo lugar (ex.: o modelo diz `helmet` onde o backend confirmou `no_helmet`), mantemos as duas e contamos um **conflito**. Você decide na revisão, e essas imagens aparecem primeiro na fila.

### IoU (Intersection over Union)
É a área da interseção dividida pela área da união de duas caixas: 1 = idênticas, 0 = não se tocam. É o conceito central para comparar caixas, e também aparece nas métricas (o "50" do mAP50 é IoU ≥ 0,5).

### Limiar de confiança: o trade-off da revisão
- **Limiar baixo** (ex.: 0,25): mais sugestões, incluindo falsos positivos. Você vai **apagar** mais caixas.
- **Limiar alto** (ex.: 0,6): menos lixo, mas o modelo deixa passar objetos. Você vai **desenhar** mais caixas.

Apagar uma caixa leva 1 segundo; desenhar uma leva 5 a 10. Por isso, em pré-anotação, costuma compensar um limiar mais baixo do que o de produção. Teste abaixo e escolha.

In [ ]:
# 🔁 Funções de pré-anotação
CONF_PRE = 0.25     # ✍️ ajuste depois de olhar o teste abaixo
IOU_DUPLICATA = 0.5
FAIXA_INCERTA = (CONF_PRE, 0.5)   # sugestões nessa faixa são as mais prováveis de estarem erradas


def sugestoes(resultado):
    """Converte a saída do ultralytics em caixas (classe_final, x, y, w, h, conf)."""
    b = resultado.boxes
    caixas = []
    for (x, y, w, h), c, conf in zip(b.xywhn.tolist(), b.cls.tolist(), b.conf.tolist()):
        novo = MAPA_ANTIGO_ID.get(int(c))
        if novo is not None:
            caixas.append((novo, x, y, w, h, conf))
    return caixas


def mesclar(gt, preds, iou_dup=IOU_DUPLICATA):
    """Caixas confirmadas + sugestões. Duplicatas da mesma classe saem; sobreposições de outra classe ficam (conflito)."""
    mantidas, conflitos = [], 0
    for p in sorted(preds, key=lambda p: -p[5]):
        sobrepostas = [g for g in gt if iou(g[1:5], p[1:5]) > iou_dup]
        if any(g[0] == p[0] for g in sobrepostas):
            continue
        conflitos += bool(sobrepostas)
        mantidas.append(p)
    return list(gt) + mantidas, mantidas, conflitos

In [ ]:
# Teste numa amostra pequena: à esquerda o rótulo original, à direita o que vai para revisão
N = 6
stems_teste = df_orig.stem.drop_duplicates().sample(N, random_state=None).tolist()
# uma imagem por chamada: passar uma lista faz o ultralytics montar UM lote com todas elas
resultados = [modelo_antigo.predict(str(DATASET_ORIGINAL / "images" / f"{s}.jpg"),
                                    conf=CONF_PRE, imgsz=IMGSZ_ANTIGO, verbose=False)[0] for s in stems_teste]

fig, axs = grade(N * 2, cols=2)
for i, (stem, r) in enumerate(zip(stems_teste, resultados)):
    gt = gt_final(stem)
    todas, novas, conflitos = mesclar(gt, sugestoes(r))
    img = DATASET_ORIGINAL / "images" / f"{stem}.jpg"
    desenhar(img, gt, CLASSES_FINAIS, axs[2 * i], f"{stem} | original")
    desenhar(img, todas, CLASSES_FINAIS, axs[2 * i + 1], f"+{len(novas)} sugestões, {conflitos} conflito(s)")
plt.tight_layout()

> ✍️ **Exercício 2.** Rode a célula de teste algumas vezes com `CONF_PRE` em 0,15, 0,25 e 0,5 (redefina a variável e rode a célula de funções de novo). Para cada valor, conte mentalmente quantas caixas você teria que apagar e quantas teria que desenhar. Qual limiar minimiza o *seu* trabalho?

Quando estiver satisfeito, rode a pré-anotação completa. Na GPU, com a YOLOv8n, leva cerca de 1 minuto para as 5.321 fotos.

In [ ]:
# ⚠️ Pré-anotação completa: gera work/revisao/{images,labels}
SOBRESCREVER = False   # proteção: não apaga uma revisão que você já começou

img_dir, lbl_dir = REVISAO / "images", REVISAO / "labels"
if lbl_dir.exists() and any(lbl_dir.iterdir()) and not SOBRESCREVER:
    raise RuntimeError(f"{lbl_dir} já tem rótulos. Use SOBRESCREVER = True se quiser gerar de novo.")
lbl_dir.mkdir(parents=True, exist_ok=True)
(REVISAO / "conf").mkdir(exist_ok=True)   # confiança de cada caixa, mostrada no revisor web

imagens = sorted((DATASET_ORIGINAL / "images").glob("*.jpg"))
for img in imagens:
    linkar(img, img_dir / img.name)

origem = df_orig.drop_duplicates("stem").set_index("stem").classe   # classe que o backend confirmou
relatorio = []
# passando a PASTA, o ultralytics lê uma imagem por vez; stream=True não guarda tudo na memória
stream = modelo_antigo.predict(str(DATASET_ORIGINAL / "images"), stream=True,
                               conf=CONF_PRE, imgsz=IMGSZ_ANTIGO, verbose=False)
for i, r in enumerate(stream, 1):
    stem = Path(r.path).stem
    gt = gt_final(stem)
    todas, novas, conflitos = mesclar(gt, sugestoes(r))
    escrever_labels(lbl_dir / f"{stem}.txt", todas)
    (REVISAO / "conf" / f"{stem}.txt").write_text("".join("gt\n" if len(b) == 5 else f"{b[5]:.4f}\n" for b in todas))
    confs = [p[5] for p in novas]
    relatorio.append(dict(
        stem=stem,
        classe_original=origem[stem],
        sugestoes=len(novas),
        incertas=sum(FAIXA_INCERTA[0] <= c < FAIXA_INCERTA[1] for c in confs),
        conflitos=conflitos,
        conf_min=min(confs) if confs else None,
    ))
    if i % 500 == 0 or i == len(imagens):
        print(f"{i}/{len(imagens)}")

(REVISAO / "classes.txt").write_text("\n".join(CLASSES_FINAIS) + "\n")
(lbl_dir / "classes.txt").write_text("\n".join(CLASSES_FINAIS) + "\n")  # algumas ferramentas procuram aqui

df_pre = pd.DataFrame(relatorio).sort_values(["conflitos", "incertas", "sugestoes"], ascending=False)
df_pre.to_csv(REVISAO / "prioridade_revisao.csv", index=False)
print("Imagens com sugestões novas:", (df_pre.sugestoes > 0).sum(), "de", len(df_pre))
print("Imagens com conflito:", (df_pre.conflitos > 0).sum())

In [ ]:
# 🔁 Quantas caixas a pré-anotação acrescentou, por classe
contagem_gt = Counter(CLASSES_FINAIS[c] for s in df_orig.stem.unique() for c, *_ in gt_final(s))
contagem_pre = Counter(CLASSES_FINAIS[c] for t in (REVISAO / "labels").glob("*.txt")
                       if t.name != "classes.txt" for c, *_ in ler_labels(t))
pd.DataFrame({"original": contagem_gt, "pré-anotação": contagem_pre}).fillna(0).astype(int)

In [ ]:
# As imagens mais "suspeitas" (conflitos primeiro): comece a revisão por elas
df_pre = pd.read_csv(REVISAO / "prioridade_revisao.csv")
topo = df_pre.head(6)
fig, axs = grade(len(topo))
for (_, linha), ax in zip(topo.iterrows(), axs):
    desenhar(REVISAO / "images" / f"{linha.stem}.jpg", ler_labels(REVISAO / "labels" / f"{linha.stem}.txt"),
             CLASSES_FINAIS, ax, f"{linha.stem} | conflitos={linha.conflitos} incertas={linha.incertas}")
plt.tight_layout()
df_pre.head(10)

## 5. Revisão (a parte manual e a mais importante)

### Antes de abrir a ferramenta: escreva o manual de anotação
Um texto curto, uma linha por classe, dizendo **o que** a caixa deve cobrir e os casos de borda. Exemplo:

| classe | a caixa cobre | casos de borda |
|---|---|---|
| no_helmet | a cabeça do motociclista/garupa | cabeça cortada na borda da foto: rotula se ≥ 50% visível |
| no_seatbelt | tronco do motorista | vidro muito escuro, não dá para ver: **não** rotula |
| ... | ... | ... |

Consistência importa mais que perfeição. O modelo aprende o padrão que você seguir. Se o padrão muda no meio do caminho, ele aprende ruído.

### Ferramenta: o revisor web deste projeto

A pasta `revisor/` tem uma interface web local para revisar as pré-anotações sem sair do seu computador. Ela lê as sugestões de `work/revisao` e grava cada imagem revisada em `work/revisado/labels`, que é exatamente onde a validação abaixo procura.

- **Uma imagem só conta como revisada depois que você salva** (`Enter` salva e passa para a próxima). As não salvas ficam fora do dataset final.
- **Caixa com ✓ e linha grossa** foi confirmada pelo backend. **Linha tracejada** é sugestão com confiança abaixo de 0,5: a mais provável de estar errada.
- **Filtros:** status, classe confirmada, "contém a classe" e "só com conflito". Revisar uma classe por vez (ex.: todas as fotos com `sandals_infraction`) deixa o olho mais rápido e as decisões mais consistentes.
- Atalhos no botão **Atalhos (?)**. Os principais: arrastar no vazio desenha; clicar seleciona; `1`…`0`, `Q`, `W`, `E` trocam a classe; `Del` apaga; `Ctrl+Z` desfaz; a roda do mouse dá zoom.

As confianças das sugestões só aparecem se a pré-anotação tiver gravado `work/revisao/conf/`. Se você rodou a pré-anotação antes dessa versão do notebook e ainda não revisou nada, rode-a de novo com `SOBRESCREVER = True` (leva 1 minuto).

> **Alternativa: CVAT ou X-AnyLabeling.** As células de lotes e de importação do CVAT continuam mais abaixo e gravam no mesmo lugar (`work/revisado/labels`), então dá para misturar as ferramentas. O [X-AnyLabeling](https://github.com/CVHub520/X-AnyLabeling) abre `work/revisao/` direto; nesse caso, marque `REVISEI_DIRETO_EM_WORK_REVISAO = True` na validação.


In [ ]:
# 🔁 Abre o revisor web num processo separado (o notebook continua livre enquanto você revisa)
import subprocess, sys, time, urllib.request

PORTA_REVISOR = 8765
URL_REVISOR = f"http://127.0.0.1:{PORTA_REVISOR}"


def revisor_no_ar():
    try:
        urllib.request.urlopen(URL_REVISOR + "/api/config", timeout=1)
        return True
    except OSError:
        return False


if revisor_no_ar():
    print("O revisor já está rodando:", URL_REVISOR)
else:
    log_revisor = open(WORK / "revisor.log", "a", encoding="utf-8")
    revisor = subprocess.Popen([sys.executable, "-m", "revisor", "--revisao", str(REVISAO), "--revisado", str(REVISADO),
                                "--porta", str(PORTA_REVISOR)], cwd=ROOT, stdout=log_revisor, stderr=subprocess.STDOUT)
    for _ in range(40):
        if revisor_no_ar() or revisor.poll() is not None:
            break
        time.sleep(0.25)
    if revisor_no_ar():
        print("Revisor aberto em", URL_REVISOR)
    else:
        print("O revisor não subiu. Veja o erro em", WORK / "revisor.log")

In [ ]:
# Fecha o revisor (as revisões já estão gravadas em disco a cada Enter)
if "revisor" in globals() and revisor.poll() is None:
    revisor.terminate()
    print("Revisor fechado.")
else:
    print("Nenhum revisor aberto por este kernel. Se ele continuar no ar, feche o processo 'python -m revisor'.")

In [ ]:
# ⚠️ (Alternativa ao revisor web) Prepara lotes para o CVAT
TAMANHO_LOTE = 500
ORDEM = "prioridade"   # "prioridade" (conflitos/incertas primeiro) ou "classe" (uma classe por vez)

df_pre = pd.read_csv(REVISAO / "prioridade_revisao.csv")
if ORDEM == "classe":
    df_pre = df_pre.sort_values(["classe_original", "stem"])
stems = df_pre.stem.tolist()

CVAT = WORK / "cvat"
for n, inicio in enumerate(range(0, len(stems), TAMANHO_LOTE), 1):
    lote = stems[inicio:inicio + TAMANHO_LOTE]
    pasta = CVAT / f"lote_{n:02d}"
    for s in lote:
        linkar(REVISAO / "images" / f"{s}.jpg", pasta / f"{s}.jpg")
    with zipfile.ZipFile(CVAT / f"lote_{n:02d}_anotacoes.zip", "w") as z:
        z.writestr("obj.names", "\n".join(CLASSES_FINAIS) + "\n")
        z.writestr("obj.data", f"classes = {len(CLASSES_FINAIS)}\nnames = data/obj.names\n"
                               "train = data/train.txt\nbackup = backup/\n")
        z.writestr("train.txt", "".join(f"data/obj_train_data/{s}.jpg\n" for s in lote))
        for s in lote:
            z.write(REVISAO / "labels" / f"{s}.txt", f"obj_train_data/{s}.txt")
    print(f"lote_{n:02d}: {len(lote)} imagens")

print("\nJSON para a aba Raw das labels do projeto no CVAT:")
print(json.dumps([{"name": n, "type": "rectangle", "attributes": []} for n in CLASSES_FINAIS], indent=2))

In [ ]:
# 🔁 Importa as exportações do CVAT (work/cvat_export/*.zip) para work/revisado/labels
EXPORT = WORK / "cvat_export"
(REVISADO / "labels").mkdir(parents=True, exist_ok=True)

importadas = 0
for zip_path in sorted(EXPORT.glob("*.zip")):
    with zipfile.ZipFile(zip_path) as z:
        nomes_cvat = z.read("obj.names").decode("utf-8").split()
        remap = {i: CLASSES_FINAIS.index(n) for i, n in enumerate(nomes_cvat)}  # por nome, não por posição
        for arq in z.namelist():
            if arq.startswith("obj_train_data/") and arq.endswith(".txt"):
                caixas = []
                for linha in z.read(arq).decode("utf-8").splitlines():
                    p = linha.split()
                    if p:
                        caixas.append((remap[int(p[0])], *map(float, p[1:5])))
                escrever_labels(REVISADO / "labels" / f"{Path(arq).stem}.txt", caixas)
                importadas += 1
print(f"{importadas} arquivos de rótulo importados de {len(list(EXPORT.glob('*.zip')))} zip(s)")
if not importadas:
    print("Nenhuma exportação em work/cvat_export ainda. É o esperado antes de revisar o primeiro lote.")

### Validação da revisão

Antes de treinar, conferimos:
- rótulos corrompidos (classe inexistente, coordenadas fora de 0–1);
- imagens **sem nenhuma caixa** (toda foto tem ao menos a infração confirmada, então isso é suspeito; nas fotos de `WRONG_WAY`, cuja caixa foi descartada, espere pelo menos o `car_front`/`car_back` do veículo);
- imagens onde a **caixa confirmada sumiu**. Pode ser correção legítima (a classe estava errada) ou uma exclusão sem querer. Vale olhar uma a uma.

In [ ]:
# 🔁 Qual pasta de rótulos usar no dataset final
REVISEI_DIRETO_EM_WORK_REVISAO = False   # ✍️ True só se revisou editando work/revisao/labels no lugar (ex.: X-AnyLabeling)

tem_revisado = (REVISADO / "labels").exists() and any((REVISADO / "labels").glob("*.txt"))
if tem_revisado:
    LABELS_REVISADAS = REVISADO / "labels"
elif REVISEI_DIRETO_EM_WORK_REVISAO:
    LABELS_REVISADAS = REVISAO / "labels"
else:
    raise RuntimeError("Ainda não há rótulos revisados: revise ao menos uma imagem no revisor web (ou exporte um "
                       "lote do CVAT e rode a célula de importação). Esta parada é de propósito: sem ela, o dataset "
                       "final seria montado com as sugestões sem revisão do modelo antigo.")
print("Usando:", LABELS_REVISADAS)

stems_todos = sorted(p.stem for p in (REVISAO / "images").glob("*.jpg"))
problemas, vazias, gt_sumiu, faltando = [], [], [], []
contagem_rev = Counter()
for s in stems_todos:
    txt = LABELS_REVISADAS / f"{s}.txt"
    if not txt.exists():
        faltando.append(s)
        continue
    caixas = ler_labels(txt)
    if not caixas:
        vazias.append(s)
    for c, x, y, w, h in caixas:
        if not 0 <= c < len(CLASSES_FINAIS):
            problemas.append((s, f"classe {c} inexistente"))
        elif not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
            problemas.append((s, f"coordenadas fora do intervalo: {x, y, w, h}"))
        else:
            contagem_rev[CLASSES_FINAIS[c]] += 1
    for g in gt_final(s):
        if not any(c[0] == g[0] and iou(c[1:5], g[1:5]) > 0.5 for c in caixas):
            gt_sumiu.append(s)

print(f"Ainda sem revisão (ficam fora do dataset final): {len(faltando)} | vazias: {len(vazias)} | "
      f"rótulos inválidos: {len(problemas)} | caixa confirmada sumiu/mudou: {len(gt_sumiu)}")
pd.DataFrame({"original": contagem_gt, "pré-anotação": contagem_pre, "revisado": contagem_rev}).fillna(0).astype(int)

In [ ]:
# Olhe as imagens onde a caixa confirmada sumiu ou mudou (esquerda = original, direita = revisado)
amostra = gt_sumiu[:4]
fig, axs = grade(len(amostra) * 2 or 1, cols=2)
for i, s in enumerate(amostra):
    img = REVISAO / "images" / f"{s}.jpg"
    desenhar(img, gt_final(s), CLASSES_FINAIS, axs[2 * i], f"{s} | original")
    desenhar(img, ler_labels(LABELS_REVISADAS / f"{s}.txt"), CLASSES_FINAIS, axs[2 * i + 1], "revisado")
plt.tight_layout()

## 6. Separação treino / validação

O modelo **nunca** pode ver as imagens de validação durante o treino. Elas são a "prova" que mede se ele generaliza ou só decorou.

- **Estratificado:** separamos 15% de *cada classe*. Sem isso, as ~60 fotos de `cellphone_infraction` podiam cair quase todas no treino (ou quase todas na validação) só por azar.
- **Vazamento (*data leakage*):** fotos da mesma câmera, tiradas em sequência, são quase idênticas. Se uma cai no treino e a "gêmea" na validação, a métrica fica otimista demais. O `metadata.csv` não tem câmera nem data. Se o backend tiver esses campos, o ideal é separar por câmera ou por dia, e não por foto.

### 6.1 Juntando o dataset extra (já rotulado com todas as classes)

O dataset extra não passa pela pré-anotação nem pela revisão: ele já está completo. Ele entra **aqui**, na hora de montar o dataset final. Três cuidados:

- **Por que juntar: esquecimento catastrófico.** Se o fine-tuning usar só as 5.321 fotos novas, o modelo se ajusta a elas e pode perder o que sabia sobre situações que só apareciam nos dados antigos. Misturar os dados antigos no treino evita isso.
- **Onde cada parte entra.** O treino do extra vai para o treino final. A validação principal continua sendo **só a deste dataset revisado**, porque é ela que representa o que chega da produção. A validação do extra vira um **teste de esquecimento** separado (seção 8): o modelo novo precisa continuar indo bem nela.
- **Fotos repetidas.** Se a mesma foto estiver nos dois datasets, ela pode acabar no treino por um lado e na validação pelo outro, e a nota fica falsamente alta. A checagem abaixo compara o conteúdo dos arquivos, e quando acha uma repetida, fica com a versão revisada deste dataset.

As classes do extra são traduzidas **pelo nome**. Se ele usar outros nomes para as mesmas coisas, preencha o `MAPA_EXTRA`.

Se o extra for muito maior que este dataset, ele domina o treino. O `FRACAO_EXTRA` permite usar só uma parte dele.

In [ ]:
# 🔁 ✍️ Dataset extra: a pasta que contém o data.yaml dele (ou None para não usar)
from IPython.display import display

DATASET_EXTRA = None   # ex.: Path(r"D:\datasets\traffic_fiscalizacao_balanced")
FRACAO_EXTRA = 1.0     # fração do treino do extra que entra no treino final
MAPA_EXTRA = {}        # nome no extra -> nome em CLASSES_FINAIS (ou None), se os nomes diferirem
EXTENSOES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def label_de(img):
    """Regra do ultralytics: troca a última pasta 'images' do caminho por 'labels'."""
    partes = list(img.parts)
    i = len(partes) - 1 - partes[::-1].index("images")
    partes[i] = "labels"
    return Path(*partes).with_suffix(".txt")


def listar_yolo(raiz):
    """Lê um dataset YOLO -> ({id: nome}, [(imagem, label, parte), ...])."""
    raiz = Path(raiz)
    with open(raiz / "data.yaml", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    nomes = cfg["names"] if isinstance(cfg["names"], dict) else dict(enumerate(cfg["names"]))
    base = Path(cfg.get("path") or ".")
    base = base if base.is_absolute() else raiz / base
    if not base.exists():
        base = raiz   # data.yaml gerado noutra máquina (ex.: caminho Linux)
    itens, vistos = [], set()
    for parte in ("train", "val", "test"):
        entradas = cfg.get(parte) or []
        for e in [entradas] if isinstance(entradas, str) else entradas:
            pasta = base / e
            if pasta.suffix == ".txt":
                raise NotImplementedError(f"'{parte}' aponta para uma lista de arquivos ({pasta}); aqui só tratamos pastas")
            for img in sorted(pasta.rglob("*")):
                if img.suffix.lower() in EXTENSOES and img not in vistos:
                    vistos.add(img)
                    itens.append((img, label_de(img), parte))
    return nomes, itens


ids_extra, itens_extra = {}, []
if DATASET_EXTRA:
    nomes_extra, itens_extra = listar_yolo(DATASET_EXTRA)
    for i, n in nomes_extra.items():
        destino = resolver(n, MAPA_EXTRA)
        ids_extra[i] = CLASSES_FINAIS.index(destino) if destino else None
        print(f"extra {i:>2} {n:<28} -> {destino or '(descartada)'}")

    contagem, sem_label = Counter(), Counter()
    for img, lbl, parte in itens_extra:
        if not lbl.exists():
            sem_label[parte] += 1
            continue
        for c, *_ in ler_labels(lbl):
            contagem[(nomes_extra[c], parte)] += 1
    print("\nImagens por parte:", dict(Counter(p for *_, p in itens_extra)))
    print("Imagens sem .txt (viram 'fundo', o que é legítimo se não houver nada nelas):", dict(sem_label))
    display(pd.Series(contagem).unstack(fill_value=0))

In [ ]:
# 🔁 Fotos repetidas entre o extra e este dataset (compara o conteúdo, não o nome)
def md5(caminho):
    return hashlib.md5(Path(caminho).read_bytes()).hexdigest()


DUPLICADAS = set()
if DATASET_EXTRA:
    # só calcula o hash de arquivos com tamanho igual a algum nosso: evita ler o dataset extra inteiro
    por_tamanho = defaultdict(list)
    for p in (REVISAO / "images").glob("*.jpg"):
        por_tamanho[p.stat().st_size].append(p)
    candidatos = [img for img, _, _ in itens_extra if img.stat().st_size in por_tamanho]
    hashes_nossos = {md5(p) for img in candidatos for p in por_tamanho[img.stat().st_size]}
    DUPLICADAS = {img for img in candidatos if md5(img) in hashes_nossos}
    print(f"{len(DUPLICADAS)} fotos do extra são idênticas a fotos deste dataset e serão ignoradas")
    print("(cópias recomprimidas ou redimensionadas não são pegas por esta checagem)")

In [ ]:
# ⚠️ Monta datasets/infractions-v2 (pode rodar de novo à vontade: é tudo derivado)
# Hard links só funcionam no mesmo disco: se o extra estiver noutro, as imagens dele são copiadas.
FRACAO_VAL = 0.15

origem_por_stem = df_orig.drop_duplicates("stem").set_index("stem").classe
por_classe = defaultdict(list)
for s in stems_todos:
    por_classe[origem_por_stem[s]].append(s)

rng = random.Random(SEED)
split = {}
for classe, lista in sorted(por_classe.items()):
    lista = sorted(lista)
    rng.shuffle(lista)
    n_val = max(1, round(len(lista) * FRACAO_VAL))
    split.update({s: "val" for s in lista[:n_val]})
    split.update({s: "train" for s in lista[n_val:]})

# O sorteio acima usa TODAS as fotos, então cada uma cai sempre do mesmo lado, não importa quantos lotes
# já foram revisados. Só agora filtramos: entram no dataset apenas as fotos com rótulo revisado.
split = {s: parte for s, parte in split.items() if (LABELS_REVISADAS / f"{s}.txt").exists()}
print(f"{len(split)} de {len(stems_todos)} fotos revisadas entram no dataset final")

if DATASET_FINAL.exists():
    shutil.rmtree(DATASET_FINAL)   # apaga só os links; as imagens originais continuam intactas
for s, parte in split.items():
    linkar(REVISAO / "images" / f"{s}.jpg", DATASET_FINAL / "images" / parte / f"{s}.jpg")
    (DATASET_FINAL / "labels" / parte).mkdir(parents=True, exist_ok=True)
    escrever_labels(DATASET_FINAL / "labels" / parte / f"{s}.txt", ler_labels(LABELS_REVISADAS / f"{s}.txt"))

EXTRA_VAL = DATASET_FINAL.parent / "extra-val"   # validação do extra = teste de esquecimento
if EXTRA_VAL.exists():
    shutil.rmtree(EXTRA_VAL)
n_extra = Counter()
if DATASET_EXTRA:
    treino_extra = [(i, l) for i, l, p in itens_extra if p == "train" and i not in DUPLICADAS]
    treino_extra = rng.sample(treino_extra, round(len(treino_extra) * FRACAO_EXTRA))
    val_extra = [(i, l) for i, l, p in itens_extra if p != "train" and i not in DUPLICADAS]
    for destino, parte, lista in ((DATASET_FINAL, "train", treino_extra), (EXTRA_VAL, "val", val_extra)):
        (destino / "labels" / parte).mkdir(parents=True, exist_ok=True)
        for img, lbl in lista:
            nome = f"extra_{img.stem}_{hashlib.md5(str(img).encode()).hexdigest()[:6]}"  # evita nomes repetidos
            linkar(img, destino / "images" / parte / f"{nome}{img.suffix}")
            caixas = [(ids_extra[c], *xywh) for c, *xywh in ler_labels(lbl) if ids_extra.get(c) is not None]
            escrever_labels(destino / "labels" / parte / f"{nome}.txt", caixas)
            n_extra[parte] += 1
    if n_extra["val"]:
        (EXTRA_VAL / "data.yaml").write_text(yaml.safe_dump(
            {"path": str(EXTRA_VAL), "train": "images/val", "val": "images/val",
             "names": dict(enumerate(CLASSES_FINAIS))}, allow_unicode=True), encoding="utf-8")
    print(f"Extra: {n_extra['train']} fotos no treino | {n_extra['val']} no teste de esquecimento")

DATA_YAML = DATASET_FINAL / "data.yaml"
with open(DATA_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump({"path": str(DATASET_FINAL), "train": "images/train", "val": "images/val",
                    "names": dict(enumerate(CLASSES_FINAIS))}, f, allow_unicode=True, sort_keys=False)

pd.crosstab(pd.Series(origem_por_stem[list(split)].values, name="classe original"),
            pd.Series(list(split.values()), name="split"))

In [ ]:
# 🔁 (se reabriu o notebook depois do split, só redefine o caminho)
DATA_YAML = DATASET_FINAL / "data.yaml"
EXTRA_VAL = DATASET_FINAL.parent / "extra-val"
print(DATA_YAML.read_text(encoding="utf-8"))

## 7. Fine-tuning

Carregamos o **modelo antigo** e chamamos `.train()` com o dataset novo. Os parâmetros que mais importam:

| parâmetro | o que é | por que esse valor |
|---|---|---|
| `epochs` | quantas passadas pelo dataset | 60 é um bom começo para fine-tuning; o `patience` interrompe antes se parar de melhorar |
| `patience` | épocas sem melhora na validação antes de parar | evita gastar tempo e *overfitting* |
| `imgsz` | lado da imagem de entrada | igual ao do modelo antigo preserva o que ele aprendeu sobre escala; maior ajuda objetos pequenos, mas custa memória |
| `batch` | imagens processadas por passo | `-1` = o ultralytics mede a VRAM e escolhe; com 6 GB, se der *out of memory*, use 4 ou 8 fixo |
| `freeze` | congela as N primeiras camadas | útil com poucos dados: só a parte final aprende. Veja o exercício 4 |
| `workers` | processos que carregam imagens | no Windows, se travar ou der erro de multiprocessing, use 0 |

Enquanto treina, o ultralytics imprime por época: as *losses* (`box_loss` = erro na posição da caixa, `cls_loss` = erro na classe, `dfl_loss` = refinamento das bordas) e as métricas de validação. **Losses de treino caindo e mAP de validação subindo = está aprendendo.** Se a loss de treino continuar caindo e o mAP de validação estagnar ou cair, é *overfitting*.

Tudo fica salvo em `runs/<nome>/`: pesos (`weights/best.pt` e `last.pt`), gráficos e `results.csv`.

In [ ]:
# ⚠️ Treino (umas 2–3 h para 60 épocas nesta GPU: deixe rodando)
EXPERIMENTO = "ft_modelo_antigo"
IMGSZ = IMGSZ_ANTIGO

modelo = YOLO(MODELO_ANTIGO)
modelo.train(
    data=str(DATA_YAML),
    epochs=60,
    patience=15,
    imgsz=IMGSZ,
    batch=-1,
    workers=4,
    seed=SEED,
    project=str(RUNS),
    name=EXPERIMENTO,
    plots=True,
)
RUN_DIR = Path(modelo.trainer.save_dir)
print("Resultados em:", RUN_DIR)

Repare na linha `Transferred X/Y items from pretrained weights` no começo do log. Aqui deve aparecer 355/355: mesmas classes, mesma arquitetura, todos os pesos reaproveitados. Se um dia você adicionar ou reordenar classes, o ultralytics casa as classes pelo nome e copia a linha da camada de classificação de cada uma (`cls_remap=True`); só as classes novas começam aleatórias. Se aparecer bem menos que 355, algo não casou (nome de classe diferente, arquitetura diferente) e vale investigar antes de treinar horas.

**Se o treino for interrompido** (queda de energia, fechou o notebook), dá para continuar de onde parou:
```python
YOLO(RUNS / EXPERIMENTO / "weights" / "last.pt").train(resume=True)
```

In [ ]:
# 🔁 Curvas de treino: losses e métricas por época
from IPython.display import Image as Img, display

EXPERIMENTO = "ft_modelo_antigo"
IMGSZ = IMGSZ_ANTIGO
RUN_DIR = RUNS / EXPERIMENTO   # ajuste se o ultralytics criou "ft_modelo_antigo2", etc.
display(Img(filename=str(RUN_DIR / "results.png"), width=1100))

> ✍️ **Exercício 3: o modelo antigo ajudou mesmo?** Treine um segundo experimento idêntico, mas partindo de um modelo genérico pré-treinado no COCO em vez do antigo (troque `YOLO(MODELO_ANTIGO)` por `YOLO("yolov8n.pt")` e `EXPERIMENTO = "base_coco"`. É a mesma arquitetura; só muda o ponto de partida: objetos genéricos do COCO x o seu modelo de fiscalização). Compare os mAPs na seção 8. Esse é o jeito honesto de medir o valor do fine-tuning.
>
> ✍️ **Exercício 4:** teste `freeze=10` (congela o *backbone*, a parte que extrai características visuais). O treino fica mais rápido e usa menos VRAM. O mAP ficou melhor ou pior? O que isso diz sobre o quanto o seu dataset difere do que o modelo antigo já tinha visto?

## 8. Avaliação

### As métricas
- **Precisão**: das detecções que o modelo fez, quantas estavam certas. Precisão baixa = multas indevidas.
- **Recall**: dos objetos que existiam, quantos o modelo achou. Recall baixo = infrações que passam batido.
- **AP** (*Average Precision*): área sob a curva precisão × recall de uma classe. Resume o desempenho em todos os limiares de confiança.
- **mAP50**: média do AP de todas as classes, considerando acerto quando IoU ≥ 0,5 com a caixa real.
- **mAP50-95**: média do mAP em IoU de 0,5 a 0,95. É mais exigente com a precisão da caixa, e é o que o ultralytics usa para escolher o `best.pt`.

Para fiscalização, precisão costuma pesar mais: um falso positivo vira uma multa indevida. Na prática, você escolhe um **limiar de confiança** de produção olhando `P_curve.png`, `R_curve.png` e `F1_curve.png`.

In [ ]:
# Avaliação do modelo novo, por classe
modelo_novo = YOLO(RUN_DIR / "weights" / "best.pt")
met_novo = modelo_novo.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, plots=True,
                           project=str(RUNS), name=f"{EXPERIMENTO}_val")


def tabela_por_classe(met, nomes):
    linhas = []
    for i, c in enumerate(met.box.ap_class_index):
        p, r, ap50, ap = met.box.class_result(i)
        linhas.append(dict(classe=nomes[int(c)], precisao=p, recall=r, mAP50=ap50, mAP50_95=ap))
    linhas.append(dict(classe="TODAS", precisao=met.box.mp, recall=met.box.mr,
                       mAP50=met.box.map50, mAP50_95=met.box.map))
    return pd.DataFrame(linhas).set_index("classe").round(3)


tab_novo = tabela_por_classe(met_novo, modelo_novo.names)
tab_novo

In [ ]:
# Matriz de confusão: quais classes o modelo confunde entre si (e com o fundo)
display(Img(filename=str(Path(met_novo.save_dir) / "confusion_matrix_normalized.png"), width=750))

**Lendo a matriz de confusão:** cada coluna é a classe **real** e cada linha é o que o modelo **previu**. A diagonal são os acertos. A linha `background` mostra objetos que o modelo não achou (falhas de recall). A coluna `background` mostra detecções onde não havia nada (falsos positivos).

**Cuidado com a validação sem revisão.** Se você rodar a comparação antes de revisar os rótulos de validação, o modelo antigo vai parecer ótimo: os rótulos são as sugestões *dele*, então ele está sendo corrigido pelo próprio gabarito. A comparação só vale depois da revisão.

Na comparação abaixo, lembre também que o modelo antigo pode ter sido treinado com algumas dessas fotos, o que deixaria a nota dele otimista. E a classe `TODAS` só é comparável se os dois modelos tiverem as mesmas classes; olhe classe a classe.

In [ ]:
# Comparação com o modelo antigo no MESMO conjunto de validação.
# Se as listas de classes diferirem, montamos uma cópia da validação com os ids
# que cada modelo conhece e comparamos classe a classe pelo nome.
def validacao_para(nomes_modelo, destino):
    ids = {n: i for i, n in nomes_modelo.items()}
    if destino.exists():
        shutil.rmtree(destino)
    for img in (DATASET_FINAL / "images" / "val").glob("*.jpg"):
        linkar(img, destino / "images" / "val" / img.name)
        caixas = [(ids[CLASSES_FINAIS[c]], x, y, w, h)
                  for c, x, y, w, h in ler_labels(DATASET_FINAL / "labels" / "val" / f"{img.stem}.txt")
                  if CLASSES_FINAIS[c] in ids]
        (destino / "labels" / "val").mkdir(parents=True, exist_ok=True)
        escrever_labels(destino / "labels" / "val" / f"{img.stem}.txt", caixas)
    arq = destino / "data.yaml"
    arq.write_text(yaml.safe_dump({"path": str(destino), "train": "images/val", "val": "images/val",
                                   "names": dict(nomes_modelo)}, allow_unicode=True), encoding="utf-8")
    return arq


yaml_antigo = validacao_para(modelo_antigo.names, DATASET_FINAL.parent / "val_modelo_antigo")
met_antigo = modelo_antigo.val(data=str(yaml_antigo), split="val", imgsz=IMGSZ_ANTIGO,
                               project=str(RUNS), name="antigo_val")
comparacao = pd.concat({"antigo": tabela_por_classe(met_antigo, modelo_antigo.names),
                        "novo": tab_novo}, axis=1)
comparacao

In [ ]:
# Lado a lado em imagens de validação: antigo (esquerda) x novo (direita)
CONF_VIS = 0.4
N = 4
val_imgs = random.sample(sorted((DATASET_FINAL / "images" / "val").glob("*.jpg")), N)

fig, axs = grade(N * 2, cols=2)
for i, img in enumerate(val_imgs):
    for j, (m, nome, sz) in enumerate([(modelo_antigo, "antigo", IMGSZ_ANTIGO), (modelo_novo, "novo", IMGSZ)]):
        b = m.predict(str(img), conf=CONF_VIS, imgsz=sz, verbose=False)[0].boxes
        caixas = [(int(c), *xywh, conf) for xywh, c, conf in zip(b.xywhn.tolist(), b.cls.tolist(), b.conf.tolist())]
        desenhar(img, caixas, m.names, axs[2 * i + j], f"{img.stem} | {nome}")
plt.tight_layout()

### Teste de esquecimento

Os dois modelos rodam na validação do dataset extra. O modelo antigo provavelmente foi treinado com esse dataset, então a nota dele aqui vem **inflada**: ele já viu dados muito parecidos. Então não esperamos que o novo ganhe do antigo nesse teste. O que interessa é que ele **não caia muito**.

Uma queda de 1 ou 2 pontos no mAP é normal. Se a queda for grande numa classe, o modelo esqueceu essa classe. Nesse caso, dá para aumentar o `FRACAO_EXTRA`, treinar menos épocas ou usar um `lr0` menor.

In [ ]:
# Teste de esquecimento: os dois modelos na validação do dataset extra
if (EXTRA_VAL / "data.yaml").exists():
    kw = dict(data=str(EXTRA_VAL / "data.yaml"), split="val", project=str(RUNS), plots=False)
    met_e_antigo = modelo_antigo.val(imgsz=IMGSZ_ANTIGO, name="antigo_extra_val", **kw)
    met_e_novo = modelo_novo.val(imgsz=IMGSZ, name=f"{EXPERIMENTO}_extra_val", **kw)
    display(pd.concat({"antigo": tabela_por_classe(met_e_antigo, modelo_antigo.names),
                       "novo": tabela_por_classe(met_e_novo, modelo_novo.names)}, axis=1))
else:
    print("Sem dataset extra (ou ele não tem validação): nada a testar.")

> ✍️ **Exercício 5.** Olhe a tabela por classe e a matriz de confusão e responda:
> 1. Qual classe é a pior? É por falta de exemplos (veja a contagem na seção 5) ou porque ela é visualmente ambígua?
> 2. Abra `runs/<experimento>_val/val_batch0_pred.jpg` e `val_batch0_labels.jpg`. Os erros do modelo são erros *dele* ou erros de *rótulo* que escaparam na revisão?

## 9. Próximos passos

- **Ciclo de melhoria contínua (*active learning*):** quando chegarem fotos novas do backend, use o **modelo novo** para pré-anotar (volte à seção 4 trocando `MODELO_ANTIGO` pelo `best.pt`). As sugestões serão melhores e a revisão, mais rápida. Priorize as imagens em que o modelo está menos confiante: é onde ele mais aprende.
- **Versione o dataset:** guarde o `work/revisado/labels` (ou o zip do CVAT) com data. Rótulos revisados são o ativo mais caro do projeto.
- **Exportar para produção:** para substituir o `fiscalizer.onnx` sem surpresas, exporte com as mesmas opções do original (estão no metadata, seção 3):
  ```python
  modelo_novo.export(format="onnx", imgsz=640, batch=1, dynamic=False, simplify=True, opset=12)
  ```
  A saída `output0` tem formato `[1, 4 + nº de classes, 8400]`: com as mesmas 13 classes, continua `[1, 17, 8400]`, e o arquivo novo substitui o antigo sem mudar o backend. Se um dia a lista de classes mudar, o backend precisa saber.
- **O `.engine`** deve ser gerado **na máquina de produção**, a partir do ONNX novo (`yolo export model=best.pt format=engine` ou `trtexec --onnx=...`), porque ele fica preso à GPU e à versão do TensorRT onde foi compilado.